In [1]:
from __future__ import annotations
from typing import Iterable, Generator, Any
from dataclasses import dataclass, field
import numpy as np
from metasmith.models.solver import _solve_by_bounded_dfs as solve
from metasmith.models.solver import Transform, Dependency, Node

class Endpoint(Node):
    def __init__(self, properties: set[str], parents: set[Endpoint]=set()):
        x: Any = parents
        super().__init__(properties=properties, parents=x)

In [2]:
scores = np.array([5, 9, 15, 3, 1])
K = 3
_f = np.argpartition(scores, -min(K, len(scores)))[-K:]
_f.sort()
candidates = scores[_f]
candidates

array([ 5,  9, 15])

In [3]:
i = np.random.randint(0, len(candidates))
candidates[i]

np.int64(5)

In [ ]:
# maps which generated endpoints are mapped to the transform's dependencies
@dataclass
class Application:
    transform: Transform
    used: dict[Dependency, Endpoint]
    produced: dict[Dependency, Endpoint]
    iteration: int = -1
    _sig: str|None = None

    def Signature(self):
        if not self._sig: 
            self._sig = self.transform.key + "".join({e.key for e in self.used.values()})
        return self._sig

_solver_node_hash = 0
def _new_sn_hash():
    global _solver_node_hash
    _solver_node_hash+=1
    return _solver_node_hash
@dataclass
class SolverNode:
    steps: list[Application]
    _step_signatures: set[str]
    production: dict[Dependency, list[Endpoint]] # product dep to produced endpoint
    have: set[Endpoint]
    # may not be valid, holds use count
    candidate_transforms: dict[Transform, int]
    score: float | None = None
    _children: list[SolverNode] = field(default_factory=list)
    _hash: int = field(default_factory=_new_sn_hash)

    def __hash__(self) -> int:
        return self._hash

    def __eq__(self, __other: object) -> bool:
        return isinstance(__other, Node) and self._hash == __other.hash
    
@dataclass
class Solution:
    complete: bool
    steps: list[Application]
    _rough_solution: SolverNode|None
    _frontier: list[SolverNode]
    _history: list[SolverNode]
    _heuristics: dict[str, dict[str, float]]
    _iterations: int
    _relavent_transforms: list[Transform]
    
def solve_by_mcts(given: Iterable[Endpoint], transforms: Iterable[Transform], target: Transform, max_iter: int=256, seed: int=42):
    np.random.seed(seed)
    # ---
    # estimate distance of nodes to target to provide guiding metric
    # filter out nodes that don't contribute to production of targets
    
    # def estimate_distances():
    #     @dataclass
    #     class DistEstNode:
    #         ref: Transform|Dependency|Endpoint

    #     def get_children(node: DistEstNode) -> Generator[DistEstNode]:
    #         ref = node.ref
    #         if isinstance(ref, Transform):
    #             for p in ref.requires:
    #                 yield DistEstNode(p)
    #         elif isinstance(ref, Dependency):
    #             for tr in transforms:
    #                 for p in tr.produces:
    #                     if p.IsA(ref):
    #                         yield DistEstNode(tr)
    #         elif isinstance(ref, Endpoint):
    #             return None # empty iterator
    #         else:
    #             raise TypeError(f"node with unknown ref [{type(ref)}]")

    #     target_node = DistEstNode(target)
    #     distances: dict[str, float] = {target_node.ref.key: -2.0}
    #     frontier: list[DistEstNode] = [DistEstNode(target)]
    #     while len(frontier)>0:
    #         node = frontier.pop(0)
    #         child_distance = distances[node.ref.key]+1
    #         for child in get_children(node):
    #             key = child.ref.key
    #             if key in distances and distances[key] >= child_distance: continue
    #             distances[key] = child_distance
    #             frontier.append(child)
    #     return distances
    # distances = estimate_distances()
    # max_distance = max(distances.values())
    # relavent_transforms = [tr for tr in transforms if tr.key in distances]

    # ---
    # monte carlo tree search

    # produced dependency to consuming transform
    product2consumer: dict[Dependency, set[Transform]] = {}
    def _iter_consumers():
        for tr in transforms: yield tr
        yield target
    for parent in transforms:
        for child in _iter_consumers():
            if parent == child: continue
            for p in parent.produces:
                if not any(p.IsA(c) for c in child.requires): continue
                product2consumer[p] = product2consumer.get(p, set())|{child}

    # requirement prototype of consumer
    # to production prototype of producer
    demand2product: dict[Dependency, set[Dependency]] = {}
    demand2producer: dict[Dependency, set[Transform]] = {}
    given_tr = Transform()
    given_appl = Application(given_tr, used={}, produced={})
    for e in given:
        p = given_tr.AddProduct(properties=e.properties)
        given_appl.produced[p] = e
    def _iter_producers():
        for tr in transforms: yield tr
        yield given_tr
    for child in _iter_consumers():
        for parent in _iter_producers():
            if parent == child: continue
            for c in child.requires:
                found = False
                for p in parent.produces:
                    if not p.IsA(c): continue
                    demand2product[c] = demand2product.get(c, set())|{p}
                    found = True
                if found:
                    demand2producer[c] = demand2producer.get(c, set())|{parent}

    opportunity_scores: dict[Transform, int] = {}
    distance_scores: dict[Transform, int] = {}
    todo: list[tuple[Transform, int]] = [(target, -1)]
    while len(todo)>0:
        node, consumer_distance = todo.pop()
        dist = consumer_distance+1
        other_dist = distance_scores.get(node, -1)
        if dist>other_dist:
            distance_scores[node] = dist
        opportunity_scores[node] = opportunity_scores.get(node, 1)+dist
        for p in node.requires:
            for producer in demand2producer[p]:
                todo.append((producer, dist))
    relavent_transforms = [tr for tr in transforms if tr in distance_scores]
    max_distance_score = max(distance_scores.values())
    
    def _prune_irrelavent_values(d: dict, value_whitelist: set):
        for k, v in d.items():
            d[k] = value_whitelist.intersection(v)
        # for k in list(d):
        #     if len(d[k])==0: del d[k]
    rts = set(relavent_transforms)|{given_tr, target}
    _prune_irrelavent_values(product2consumer, rts)
    _prune_irrelavent_values(demand2producer, rts)
    rtsp = {p for t in rts for p in t.produces}
    _prune_irrelavent_values(demand2product, rtsp)

    @dataclass
    class ApplyResult:
        used: dict[Dependency, Endpoint]
        result: Application|None = None
    def try_apply(state: SolverNode, tr: Transform):
        used: dict[Dependency, Endpoint] = {}
        lineage: set[Endpoint] = set()

        # Transforms define lineage constraints (LC) first.
        # The endpoint matched to the LC must also be used to satisfy all instances.
        # That is, if a transform specifies A via P and B via P, 
        # then endpoint P' matched to P must be used to create both A and B
        def satisfies(e: Endpoint, p: Dependency):
            if not e.IsA(p): return False
            for parent in p.parents:
                assert isinstance(parent, Dependency)
                matched = used[parent]
                if matched not in e.parents: return False
            return True
        
        def _find_endpoint(p: Dependency):
            for product in demand2product[p]:
                if product not in state.production: continue
                for e in state.production[product]:
                    if satisfies(e, p):
                        return e
            return False
        applicable = True
        for p in tr.requires:
            e = _find_endpoint(p)
            if not e:
                applicable = False
                break
            used[p] = e
            lineage.add(e)
            for parent in e.parents:
                assert isinstance(parent, Endpoint)
                lineage.add(parent)
        if not applicable:
            application = None
        else:
            produced: dict[Dependency, Endpoint] = {}
            for p in tr.produces:
                e = Endpoint(p.properties, parents=lineage)
                produced[p] = e
            application = Application(tr, used, produced)
        return ApplyResult(
            used=used,
            result=application,
        )

    free_transforms = [t for t in relavent_transforms if len(t.requires)==0]
    def get_applicable_transforms(state: SolverNode):
        def _iter_transforms():
            for tr in state.candidate_transforms:
                yield tr
            for tr in free_transforms:
                yield tr
        # lt = None if len(state.steps)==0 else state.steps[-1].transform
        # print(lt, state.have)
        for tr in _iter_transforms():

            # **
            # ** possible for multiple endpoints per dependency
            # **
            appl = try_apply(state, tr)
            # print(appl.result is not None, distances.get(tr.key), tr)
            if appl.result:
                yield appl.result
        # print()

    def is_solved(state: SolverNode):
        last_transform = state.steps[-1].transform
        return last_transform == target

    def select_node(frontier: list[SolverNode]):
        if np.random.random() < 0.9: # exploit
            scores = np.array([s.score for s in frontier])
            K = 10
            k = min(K, scores.shape[0])
            candidate_indexes = np.argpartition(scores, -k)[-k:]
            i = np.random.choice(candidate_indexes)
        else: # explore
            i = np.random.randint(0, len(frontier))
        return i, frontier[i]

    def create_new_node(state: SolverNode, appl: Application, in_place: bool=False):
        if appl.transform in state.candidate_transforms:
            remaining_uses = state.candidate_transforms[appl.transform]
            remaining_uses -= 1
            candidate_transforms = {tr:c for tr, c in state.candidate_transforms.items() if tr != appl.transform}
            if remaining_uses>0:
                candidate_transforms[appl.transform] = remaining_uses
        else:
            candidate_transforms = state.candidate_transforms.copy() # was free transform
        for p in appl.transform.produces:
            if p not in product2consumer: continue
            for linked in product2consumer[p]:
                candidate_transforms[linked] = candidate_transforms.get(linked, 0)+1
        production = state.production.copy()
        for p, e in appl.produced.items():
            production[p] = production.get(p, [])+[e]

        if in_place:
            state.steps += [appl]
            state._step_signatures|={appl.Signature()}
            state.have |= set(appl.produced.values())
            state.candidate_transforms=candidate_transforms
            state.production = production
            new = state
        else:
            new = SolverNode(
                steps=state.steps+[appl],
                _step_signatures=state._step_signatures|{appl.Signature()},
                have=state.have|set(appl.produced.values()),
                candidate_transforms=candidate_transforms,
                production=production,
            )
        return new
    
    def merge_nodes(inbound: SolverNode, base: SolverNode, frontier: set[SolverNode]):
        # updates reciever in place with applications of giver
        def _merge(giver: SolverNode, reciever: SolverNode):
            changed = False
            for step in giver.steps:
                if step.Signature() in reciever._step_signatures: continue
                changed = True
                reciever = create_new_node(reciever, step, in_place=True)
            return changed
        
        todo: list[SolverNode] = [base]
        while len(todo)>0:
            node = todo.pop()
            if node in frontier:
                merged = _merge(inbound, node)
                if not merged: continue
                print(f"   +[{node._hash}]")
            for child in node._children:
                todo.append(child)

    def mcts(initial_node: SolverNode, max_iter: int):
        frontier: list[SolverNode] = [initial_node]
        frontier_set: set[SolverNode] = {initial_node}
        def _add_to_frontier(state: SolverNode):
            frontier.append(state)
            frontier_set.add(state)

        def _remove_from_frontier(i: int, state: SolverNode):
            _statei = i
            frontier[_statei], frontier[-1] = frontier[-1], frontier[_statei]
            frontier.pop() # O(1) vs O(m) for arr.remove()
            frontier_set.remove(state)

        application_counts: dict[str, int] = {}
        def _score(state: SolverNode):
            if len(state.steps) == 0: return 1
            last_step = state.steps[-1]
            dist = distance_scores[last_step.transform]
            prev_dist = max_distance_score if len(state.steps)<2 else distance_scores[state.steps[-2].transform]
            dist = max((prev_dist-dist)/max_distance_score, 0)
            redundancy = application_counts.get(last_step.Signature(), 0)
            redundancy = 1/(1+redundancy/10)
            opportunity = opportunity_scores[last_step.transform]
            opportunity = 1-(1/(1+opportunity/10))

            score = 1000*dist+10*opportunity+redundancy
            # score = 100*dist+10*opportunity
            return score

        i = 0
        seen: dict[str, SolverNode] = {}
        history: list[SolverNode] = []
        while len(frontier)>0:
            if i >= max_iter: break
            i += 1
            _statei, state = select_node(frontier)
            _remove_from_frontier(_statei, state)
            if len(state.steps)>0:
                last_step = state.steps[-1]
                lsk = last_step.Signature()
                application_counts[lsk] = application_counts.get(lsk, 0)+1
                print()
                print(len(history) if lsk not in seen else "-", f"[{state._hash}]", last_step.transform)
                for step in state.steps:
                    print("  ", step.transform)
                if lsk in seen:
                    original = seen[lsk]
                    print(f"  ++[{original._hash}]")
                    merge_nodes(state, original, frontier_set)
                    # redirect prior steps to original
                    for step in state.steps[:-1]:
                        seen[step.Signature()] = original
                    continue
                seen[lsk] = state
            history.append(state)
            for appl in get_applicable_transforms(state):
                if appl.Signature() in state._step_signatures: continue
                print()
                new_state = create_new_node(state, appl)
                state._children.append(new_state)
                new_state.score = _score(new_state)
                new_state.steps[-1].iteration = i
                if is_solved(new_state): return i, frontier, history, new_state
                _add_to_frontier(new_state)
        return i, frontier, history, None

    # ---
    # prune spurious nodes
    def prune(steps: list[Application]):
        e2source: dict[Endpoint, Application] = {}
        for step in steps:
            for e in step.produced.values():
                e2source[e] = step
    
        @dataclass
        class PruneNode:
            ref: Application|Endpoint

            def GetKey(self):
                if isinstance(self.ref, Application):
                    return self.ref.Signature()
                else:
                    return self.ref.key
                
            def GetChildren(self):
                if isinstance(self.ref, Application):
                    for x in self.ref.used.values():
                        yield x
                else:
                    if self.ref not in e2source: return
                    appl = e2source[self.ref]
                    yield appl

        start = PruneNode(steps[-1]) # last should be target
        todo: list[PruneNode] = [start]
        seen: dict[str, PruneNode] = {}
        while len(todo)>0:
            node = todo.pop(0)
            key = node.GetKey()
            if key in seen: continue
            seen[key] = node
            for x in node.GetChildren():
                todo.append(PruneNode(x))
        required = [x.ref for x in seen.values() if isinstance(x.ref, Application)]
        required.reverse()
        return required
    
    # ---
    # order nodes by steps to create
    def get_order(steps: list[Application]):
        seen: set[str] = set()
        _have: set[Endpoint] = {e for e in given}
        order: dict[str, int] = {e.key:0 for e in _have}
        while len(seen)<len(steps):
            changed = False
            for step in steps:
                if step.Signature() in seen: continue
                if any(e not in _have for e in step.used.values()): continue
                changed = True
                _have |= {e for e in step.produced.values()}
                seen.add(step.Signature())
                if len(step.used)>0:
                    step_depth = max(order[e.key] for e in step.used.values())+1
                else:
                    step_depth = 1
                order[step.Signature()] = step_depth
                for e in step.produced.values():
                    if e in order: continue
                    order[e.key] = step_depth+1
            if not changed: break # shouldn't happen, but here to prevent endless loop
        max_depth = max(order.values())+1
        for step in steps:
            k = step.Signature()
            if k in order: continue
            order[k] = max_depth
        return order
    
    starting_candidate_transforms: dict[Transform, int] = {} # tr, uses
    for e in given:
        for tr in relavent_transforms:
            if not any(e.IsA(p) for p in tr.requires): continue
            starting_candidate_transforms[tr] = starting_candidate_transforms.get(tr, 0)+1
    produced_by_given: dict[Dependency, list[Endpoint]] = {}
    for p, e in given_appl.produced.items():
        produced_by_given[p] = produced_by_given.get(p, [])+[e]
    iterations, frontier, history, solution = mcts(
        initial_node=SolverNode(
            steps=[],
            _step_signatures=set(),
            have=set(given),
            candidate_transforms=starting_candidate_transforms,
            production=produced_by_given
        ),
        max_iter=max_iter
    )
    D2T_KEY = "distance to target"
    d2t_report = {k.key:float(v) for k, v in distance_scores.items()}
    if solution is None:
        return Solution(
            complete=False,
            steps=[],
            _rough_solution=None,
            _frontier=frontier,
            _history=history,
            _heuristics={
                D2T_KEY: d2t_report,
            },
            _iterations=iterations,
            _relavent_transforms=relavent_transforms,
        )

    pruned_steps = prune(solution.steps)
    node_order = get_order(pruned_steps)
    ordered_steps = sorted(pruned_steps, key=lambda s: node_order[s.Signature()]*10000+len(s.used))

    return Solution(
        complete=True,
        steps=ordered_steps,
        _rough_solution=solution,
        _frontier=frontier,
        _history=history,
        _heuristics={
            "production depth": {k:float(v) for k, v in node_order.items()},
            D2T_KEY: d2t_report,
        },
        _iterations=iterations,
        _relavent_transforms=relavent_transforms,
    )

In [5]:
from enum import Enum
from pathlib import Path

from metasmith.hashing import KeyGenerator

class DAGRenderer:
    def __init__(self, plan: list[Application], meta: dict[str, Any]|None=None) -> None:
        self.plan = plan
        self.meta = meta if meta is not None else {}

    class NodeType(Enum):
        TRANSFORM = 1
        DATA      = 2
    def RenderNode(self, **meta) -> str:
        style: dict[str, str] = {}
        ntype = meta["ntype"]
        name = meta["name"]
        for k in ["style", "color"]:
            if k not in meta: continue
            style[k] = meta[k]
        match ntype:
            case self.NodeType.TRANSFORM:
                style["shape"] = "oval"
            case self.NodeType.DATA:
                style["shape"] = "box"
        style_str = " ".join(f'{k}="{v}"' for k, v in style.items())
        return f'"{name}" [{style_str}]'

    def AsDAG(self, *, font: str = 'Arial', hide_images: bool = True) -> str:
        lines = ["digraph G {"]
        lines += [f'graph [fontname="{font}"];', f'node  [fontname="{font}"];', f'edge  [fontname="{font}"];']
        plan = self.plan
        for step in plan:
            k = step.Signature()
            transform_name = f"{step.transform}:{KeyGenerator.FromStr(k, l=4)[1]}"
            meta = self.meta.get(k, {})
            if "name" in meta:
                transform_name = meta['name']
            if "prefix" in meta:
                transform_name = f"{meta['prefix']}:{transform_name}"
            meta['name'] = transform_name
            lines.append(self.RenderNode(ntype=self.NodeType.TRANSFORM, **meta))
            data_nodes = [("i", p, e) for p, e in step.used.items()]
            data_nodes += [("o", p, e) for p, e in step.produced.items()]
            for direction, p, e in data_nodes:
                name = str(e)
                meta = dict(name=name)|self.meta.get(k, {})
                lines.append(self.RenderNode(ntype=self.NodeType.DATA, **meta))
                match direction:
                    case "i":
                        lines.append(f'    "{name}" -> "{transform_name}";')
                    case "o":
                        lines.append(f'    "{transform_name}" -> "{name}";')
        lines.append("}")
        return "\n".join(lines)

    def RenderDAG(self, path_base: Path|str, format: str ='svg', *, font: str = 'Arial', hide_images: bool = True):
        import graphviz
        dag_str = self.AsDAG(font=font, hide_images=hide_images)
        src = graphviz.Source(dag_str, filename=path_base, format=format)
        src.render(cleanup=True)

In [6]:
def _tr(uses, makes):
    t = Transform()
    for x in uses:
        t.AddRequirement(properties={p.strip() for p in x.split(",")})
    for x in makes:
        t.AddProduct(properties={p.strip() for p in x.split(",")})
    return t

# ts: list[Transform] = []
# ts.append(_tr(uses = ["d0-0"], makes = ["d1-0"]))
# ts.append(_tr(uses = ["d1-0"], makes = ["d2-0"]))

# given = [
#     Endpoint(properties={"d0-0"}),
# ]

# target = Transform()
# target.AddRequirement(properties={"d2-0"})

ts: list[Transform] = []
ts.append(_tr(uses = ["start"], makes = ["a", "b"]))
ts.append(_tr(uses = ["a"], makes = ["via, v1"]))
ts.append(_tr(uses = ["b"], makes = ["via, v2"]))
ts.append(_tr(uses = ["via"], makes = ["target"]))

given = [
    Endpoint(properties={"start"}),
]

target = Transform()
p = target.AddRequirement(properties={"via"})
target.AddRequirement(properties={"target"}, parents={p})

solution = solve_by_mcts(given, ts, target)
if solution.complete:
    print(f"steps: {len(solution.steps)}")
    order = solution._heuristics["production depth"]
    for appl in solution.steps:
        depth = order[appl.Signature()]
        depth = int((depth-1)/2)
        print(depth, appl.transform)
else:
    print("failed")
    frontier = solution._frontier
    fscores = [s.score if s.score else 0 for s in frontier]
    sf = sorted(zip(frontier, fscores), key=lambda t: t[-1], reverse=True)
    for s, i in sf[:10]:
        print(i)
        for step in s.steps:
            print(step)
        print()
# renderer = DAGRenderer(solution.steps+[
renderer = DAGRenderer(solution._rough_solution.steps)
renderer.RenderDAG("./cache/mcts")

steps: 4
0 {start}->{a},{b}
1 {a}->{v1-via}
2 {via}->{target}
3 {via},{target}->


In [226]:
ts: list[Transform] = []

ts.append(_tr(uses = ["lr accession"], makes = ["lr"]))
ts.append(_tr(uses = ["lr"], makes = ["lr filtered"]))
ts.append(_tr(uses = ["lr"], makes = ["lr self map"]))
ts.append(_tr(uses = ["lr self map"], makes = ["miniasm est"]))

t = Transform()
via = t.AddRequirement(properties={"lr"})
t.AddRequirement(properties={"miniasm est"}, parents={via})
t.AddProduct(properties={"lr filtered"})
ts.append(t)

# ts.append(_tr(uses = ["lr filtered"], makes = ["assembly, lr asm"]))
t = Transform()
via = t.AddRequirement(properties={"miniasm est"})
t.AddRequirement(properties={"lr filtered"}, parents={via})
t.AddProduct(properties={"assembly", "lr asm"})
ts.append(t)

ts.append(_tr(uses = ["sr accession"], makes = ["sr"]))
ts.append(_tr(uses = ["sr"], makes = ["sr trimmed"]))
ts.append(_tr(uses = ["sr trimmed"], makes = ["assembly, sr asm"]))

ts.append(_tr(uses = ["sr trimmed", "assembly, lr asm"], makes = ["seq aln map"]))
ts.append(_tr(uses = ["seq aln map"], makes = ["bin aln map"]))
ts.append(_tr(uses = ["bin aln map", "assembly, lr asm"], makes = ["assembly, hybrid asm"]))

ts.append(_tr(uses = ["assembly"], makes = ["genes", "cds"]))

ts.append(_tr(uses = [], makes = ["bakta ref"]))
ts.append(_tr(uses = [], makes = ["busco ref", "busco map"]))
ts.append(_tr(uses = [], makes = ["cazy ref"]))
ts.append(_tr(uses = [], makes = ["kfs ko_list"]))
ts.append(_tr(uses = [], makes = ["kfs profiles"]))

t = Transform()
asm = t.AddRequirement(properties={"assembly"})
t.AddRequirement(properties={"bakta ref"})
t.AddRequirement(properties={"cds"}, parents={asm})
t.AddRequirement(properties={"genes"}, parents={asm})
t.AddProduct(properties={"bakta ann"})
ts.append(t)

ts.append(_tr(uses = ["cazy ref", "cds"], makes = ["cazy ann"]))
ts.append(_tr(uses = ["busco ref", "busco map", "cds"], makes = ["busco ann"]))
ts.append(_tr(uses = ["kfs profiles", "kfs ko_list", "cds"], makes = ["kfs ann"]))

t = Transform()
asm = t.AddRequirement(properties={"assembly"})
t.AddRequirement(properties={"bakta ann"}, parents={asm})
t.AddRequirement(properties={"cazy ann"}, parents={asm})
t.AddRequirement(properties={"busco ann"}, parents={asm})
t.AddRequirement(properties={"kfs ann"}, parents={asm})
t.AddProduct(properties={"annotations"})
ts.append(t)

t = Transform()
asm = t.AddRequirement(properties={"assembly", "hybrid asm"})
t.AddRequirement(properties={"annotations"}, parents={asm})
t.AddProduct(properties={"figures"})
ts.append(t)

given = [
    Endpoint(properties={"lr accession"}),
    Endpoint(properties={"sr accession"}),
    # Endpoint(properties={"assembly", "hybrid asm"}),
    # Endpoint(properties={"assembly", "lr asm"}),
]

target = Transform()
# p = target.AddRequirement(properties={"assembly", "sr asm"})
# p = target.AddRequirement(properties={"miniasm est"})
# p = target.AddRequirement(properties={"assembly", "lr asm"}, parents={p})
# p = target.AddRequirement(properties={"assembly", "lr asm"})
p = target.AddRequirement(properties={"assembly", "hybrid asm"})
# target.AddRequirement(properties={"genes"})
# target.AddRequirement(properties={"genes"}, parents={p})
# target.AddRequirement(properties={"bakta ann"})
# target.AddRequirement(properties={"bakta ann"}, parents={p})
# target.AddRequirement(properties={"cazy ann"}, parents={p})
# target.AddRequirement(properties={"kfs ann"})
# target.AddRequirement(properties={"kfs ann"}, parents={p})
# target.AddRequirement(properties={"annotations"})
# target.AddRequirement(properties={"annotations"}, parents={p})
# target.AddRequirement(properties={"figures"})

# %prun solution = solve_by_mcts(given, ts, target, max_iter=2**12)
solution = solve_by_mcts(given, ts, target, max_iter=2**12, seed=3)

if solution.complete:
    print(f"iters: {solution._iterations}, solution size: {len(solution.steps)}")
    order = solution._heuristics["production depth"]
    d = {}
    l = len(solution.steps)
    for i, appl in enumerate(solution.steps):
        k = appl.transform.key
        k = appl.Signature()
        depth = order[k]
        depth = int((depth-1)/2)
        d[k] = {
            "prefix": f"{depth}",
            "color": f"0,0,{0.5+((l-i)/(2*l))}",
            "style": "filled",
        }
        print(depth, appl.transform)
else:
    print("failed")
    frontier = solution._frontier
    print(len(frontier), solution._iterations)
    fscores = [s.score if s.score else 0 for s in frontier]
    sf = sorted(zip(frontier, fscores), key=lambda t: t[-1], reverse=True)
    d = solution._heuristics["distance to target"]
    d = {k:{"prefix":f"{v}"} for k, v in d.items()}
    for s, i in sf[:3]:
        print("score", i)
        for step in s.steps:
            print(step)
        print()

renderer = DAGRenderer(solution.steps, d)
renderer.RenderDAG("./cache/mcts")


1 [2] {lr accession}->{lr}
   {lr accession}->{lr}

2 [4] {sr accession}->{sr}
   {lr accession}->{lr}
   {sr accession}->{sr}

3 [9] {sr}->{sr trimmed}
   {lr accession}->{lr}
   {sr accession}->{sr}
   {sr}->{sr trimmed}

4 [8] {lr}->{lr filtered}
   {lr accession}->{lr}
   {sr accession}->{sr}
   {lr}->{lr filtered}

- [3] {sr accession}->{sr}
   {sr accession}->{sr}
  ++[4]

- [13] {sr}->{sr trimmed}
   {lr accession}->{lr}
   {sr accession}->{sr}
   {lr}->{lr filtered}
   {sr}->{sr trimmed}
  ++[9]
   +[10]

- [11] {lr}->{lr filtered}
   {lr accession}->{lr}
   {sr accession}->{sr}
   {sr}->{sr trimmed}
   {lr}->{lr filtered}
  ++[9]

- [10] {lr}->{lr filtered}
   {lr accession}->{lr}
   {sr accession}->{sr}
   {sr}->{sr trimmed}
   {lr}->{lr self map}
   {lr}->{lr filtered}
  ++[9]

- [5] {lr}->{lr self map}
   {lr accession}->{lr}
   {lr}->{lr self map}
  ++[9]

- [6] {lr}->{lr filtered}
   {lr accession}->{lr}
   {lr}->{lr filtered}
  ++[9]

- [7] {lr}->{lr self map}
   {lr ac

In [140]:
len(ts)

24

In [ ]:
print(len(solution._rough_solution.steps))
d = solution._heuristics["distance to target"]
renderer = DAGRenderer(solution._rough_solution.steps, d)
renderer.RenderDAG("./cache/mcts_raw")

In [192]:
frontier = solution._frontier
fscores = [n.score if n.score else -1 for n in frontier]
fscores = [len(n.steps) for n in frontier]
# sf = sorted(zip(frontier, fscores), key=lambda t: t[-1], reverse=True)
# picked: SolverNode = sf[3][0]
# picked: SolverNode = solution._history[100]
# print(f"score:{picked.score}")

import os
from tqdm import tqdm

Path("./cache/hist").mkdir(exist_ok=True, parents=True)
os.system(f"rm ./cache/hist/*")
todo = solution._history[:50]
for i, picked in tqdm(enumerate(todo), total=len(todo)):
    _depths = solution._heuristics["distance to target"]
    l = len(picked.steps)
    d = {a.Signature():{
        # "prefix":f"{i}/{int(_depths[a.transform.key])}/{a.iteration}",
        "prefix":f"{a.iteration}",
        "style":f"filled",
        # "color":f"0,0,{0.5+((l-i)/(2*l))}",
        "color":f"0,0,{1-0.5*(a.iteration/solution._iterations)}",
    } for i, a in enumerate(picked.steps)}
    renderer = DAGRenderer(picked.steps, d)
    renderer.RenderDAG(f"./cache/hist/{i:05}", format="png")

100%|██████████| 5/5 [00:00<00:00, 47.47it/s]
